# 03 model optimization

**Owner:** Denis. Original notebook cells: 43.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Preserve the original trained model
original_mlp = mlp_model
original_mlp_threshold = best_mlp_threshold

experiments = [
    {"name": "Smaller MLP", "layers": [32, 16], "dropout": 0.20},
    {"name": "Higher Dropout", "layers": [64, 32], "dropout": 0.30},
    {"name": "Wider MLP", "layers": [128, 64], "dropout": 0.20}
]

tuning_results = []
trained_models = {"Original MLP": original_mlp}
model_thresholds = {"Original MLP": original_mlp_threshold}

# Include original results in comparison
tuning_results.append({
    "Model": "Original MLP",
    "Threshold": original_mlp_threshold,
    **mlp_metrics
})

def build_mlp(layer_sizes, dropout):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(X_train_nn.shape[1],)))

    for units in layer_sizes:
        model.add(tf.keras.layers.Dense(units, activation="relu"))
        model.add(tf.keras.layers.Dropout(dropout))

    model.add(tf.keras.layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="roc_auc"),
            tf.keras.metrics.AUC(name="pr_auc", curve="PR")
        ]
    )
    return model

for config in experiments:
    print(f"\n=== Training {config['name']} ===")

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = build_mlp(config["layers"], config["dropout"])

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history_run = model.fit(
        X_train_nn,
        y_train_nn,
        validation_data=(X_val_nn, y_val_nn),
        epochs=100,
        batch_size=64,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=0
    )

    probabilities = model.predict(
        X_val_nn, batch_size=256, verbose=0
    ).ravel()

    # Select threshold by validation F1
    best_f1 = -1
    selected_threshold = 0.50

    for threshold in np.round(np.arange(0.20, 0.81, 0.05), 2):
        predictions = (probabilities >= threshold).astype(int)
        score = f1_score(y_val_nn, predictions, zero_division=0)

        if score > best_f1:
            best_f1 = score
            selected_threshold = float(threshold)

    predictions = (probabilities >= selected_threshold).astype(int)

    results = {
        "Model": config["name"],
        "Threshold": selected_threshold,
        "Accuracy": accuracy_score(y_val_nn, predictions),
        "Precision": precision_score(
            y_val_nn, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val_nn, predictions, zero_division=0
        ),
        "F1": f1_score(y_val_nn, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val_nn, probabilities),
        "PR-AUC": average_precision_score(
            y_val_nn, probabilities
        )
    }

    tuning_results.append(results)
    trained_models[config["name"]] = model
    model_thresholds[config["name"]] = selected_threshold

    print("Epochs:", len(history_run.history["loss"]))
    print("Best threshold:", selected_threshold)
    print("Validation F1:", round(results["F1"], 4))

# Compare all configurations
tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values("F1", ascending=False)

print("\nMLP HYPERPARAMETER COMPARISON: ")
display(tuning_df.round(4))

best_model_name = tuning_df.iloc[0]["Model"]
best_mlp_model = trained_models[best_model_name]
final_mlp_threshold = model_thresholds[best_model_name]

print("\nBest validation F1 model:", best_model_name)
print("Selected threshold:", final_mlp_threshold)
